In [ ]:
#Libraries
from ultralytics import YOLO
from roboflow import Roboflow
from pathlib import Path

In [ ]:
# Download training data from Roboflow
rf = Roboflow(api_key="44ymrV22ZuZgEYHyCQ3f")
project = rf.workspace("home-hdrxf").project("clothing-detection-yr1tn-yaxbx")
version = project.version(1)
dataset = version.download("yolov11")

: 

In [ ]:
# Create a YAML config file that combines original classes with new ones
def create_combined_yaml(dataset_path, output_path="custom_dataset.yaml"):
    """Create a custom YAML file that preserves original COCO classes and adds new ones"""
    
    # Load the dataset.yaml from Roboflow download
    roboflow_yaml = Path(dataset_path) / "data.yaml"
    
    with open(roboflow_yaml, "r") as f:
        yaml_content = f.read()
    
    # Parse the YAML content and modify it
    import yaml
    data = yaml.safe_load(yaml_content)
    
    # Manually define COCO classes since we can't import them
    COCO80_CLASSES = [
        'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 
        'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 
        'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 
        'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
        'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
        'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 
        'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
        'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 
        'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
        'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
    
    # Combine classes
    # First keep all original COCO classes
    combined_names = list(COCO80_CLASSES)
    
    # Then add new classes from Roboflow that aren't in COCO
    new_classes = data.get('names', {})
    if isinstance(new_classes, list):
        # Convert list-based classes to dict
        new_classes = {i: name for i, name in enumerate(new_classes)}
    
    # Add new classes with indices starting after the last COCO class
    next_idx = len(combined_names)
    for class_name in new_classes.values():
        if class_name not in combined_names:
            combined_names.append(class_name)
    
    # Create new YAML with combined classes
    combined_data = {
        'path': str(dataset_path),
        'train': str(Path(dataset_path) / 'train' / 'images'),
        'val': str(Path(dataset_path) / 'valid' / 'images'),
        'test': str(Path(dataset_path) / 'test' / 'images') if (Path(dataset_path) / 'test').exists() else '',
        'names': {i: name for i, name in enumerate(combined_names)},
        'nc': len(combined_names)
    }
    
    # Write the combined YAML
    with open(output_path, 'w') as f:
        yaml.dump(combined_data, f, sort_keys=False)
    
    print(f"Created combined dataset config at {output_path}")
    print(f"Total classes: {len(combined_names)}")
    return output_path

In [ ]:
# Create combined YAML file
custom_yaml = create_combined_yaml('/home/tommytang111/Projects/Drone/Clothing-Detection-1')

Created combined dataset config at custom_dataset.yaml
Total classes: 89


In [ ]:
#Load model
model = YOLO("yolo11n.pt")

100%|██████████| 5.35M/5.35M [00:00<00:00, 10.1MB/s]


In [ ]:
# Train the model with transfer learning
# Use your existing YOLO model as the starting point
epochs = 50 
batch_size = 16  
imgsz = 640

# Start training (will preserve original weights and add new classes)
results = model.train(
    data=custom_yaml,
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    patience=5,  # Early stopping patience
    device=0,     # GPU device (use 'cpu' if no GPU)
    project='yolov11_custom',  # Project name for saving results
    name='clothing_detection',  # Run name 
    pretrained=True,  # Use pretrained weights
    verbose=True  # Show training progress
)

# After training, load the best model
trained_model = YOLO(results.best)  # Use the best model from training

print(f"Model trained and saved at: {results.best}")

New https://pypi.org/project/ultralytics/8.3.102 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.96 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=custom_dataset.yaml, epochs=50, time=None, patience=15, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=yolov11_custom, name=clothing_detection, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=Non

100%|██████████| 755k/755k [00:00<00:00, 11.3MB/s]
E0000 00:00:1743878592.221127   36067 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743878592.245432   36067 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743878592.403564   36067 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743878592.403613   36067 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743878592.403614   36067 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743878592.403616   36067 computation_placer.cc:177] comput

Overriding model.yaml nc=80 with nc=89

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

train: Scanning /home/tommytang111/Projects/Drone/Clothing-Detection-1/train/labels... 840 images, 6 backgrounds, 0 corrupt: 100%|██████████| 840/840 [00:00<00:00, 1424.73it/s]

train: New cache created: /home/tommytang111/Projects/Drone/Clothing-Detection-1/train/labels.cache


...

                   all         80         75      0.921      0.906      0.944      0.823



50 epochs completed in 0.104 hours.
Optimizer stripped from yolov11_custom/clothing_detection/weights/last.pt, 5.6MB
Optimizer stripped from yolov11_custom/clothing_detection/weights/best.pt, 5.6MB

Validating yolov11_custom/clothing_detection/weights/best.pt...
Ultralytics 8.3.96 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
YOLO11n summary (fused): 100 layers, 2,629,757 parameters, 0 gradients, 6.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:00<00:00,  5.44it/s]


                   all         80         75      0.921      0.907      0.944      0.823
                person         11         11      0.905      0.872      0.906      0.767
               bicycle         10         10          1      0.564      0.769      0.664
                   car         11         11      0.924      0.727      0.851      0.741
            motorcycle          7          7      0.854          1      0.995       0.89
              airplane          9          9      0.941          1      0.995      0.881
                   bus          7          7      0.965          1      0.995      0.844
                 train          7          7      0.895          1      0.995      0.901
                 truck          6          6      0.903          1      0.995        0.9
                  boat          7          7      0.902          1      0.995       0.82
Speed: 0.2ms preprocess, 2.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to yolov11_cus

AttributeError: 'DetMetrics' object has no attribute 'best'. See valid attributes below.

    Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

    Attributes:
        save_dir (Path): A path to the directory where the output plots will be saved.
        plot (bool): A flag that indicates whether to plot precision-recall curves for each class.
        names (dict): A dictionary of class names.
        box (Metric): An instance of the Metric class for storing detection results.
        speed (dict): A dictionary for storing execution times of different parts of the detection process.
        task (str): The task type, set to 'detect'.
    